# 1.1.3 Operators

Python operators applied to real data science problems.
- Arithmetic, comparison, logical, bitwise operators
- Operator overloading with `__dunder__` methods
- Feature engineering using operators on California Housing data

In [ ]:
import urllib.request, csv
from pathlib import Path

DATA = Path('data'); DATA.mkdir(exist_ok=True)
dest = DATA / 'cal_housing.csv'
if not dest.exists():
    url = 'https://huggingface.co/datasets/scikit-learn/california-housing/resolve/main/cal_housing.csv'
    try:
        urllib.request.urlretrieve(url, dest)
        print(f'Downloaded {dest.stat().st_size} bytes')
    except Exception as e:
        print(f'Offline: {e}')
print('Done')

In [ ]:
import pandas as pd
df = pd.read_csv(dest)
print(df.shape)
df.head()

In [ ]:
# Feature engineering with arithmetic operators
df['bedroom_ratio']    = df['AveBedrms'] / df['AveRooms']       # division
df['rooms_per_person'] = df['AveRooms'] / df['AveOccup']        # division
df['is_expensive']     = df['MedHouseVal'] > 4.0                # comparison -> bool
df['price_per_room']   = df['MedHouseVal'] * 100_000 / df['AveRooms']  # multiply

print(f'Expensive houses: {df["is_expensive"].sum()} ({df["is_expensive"].mean():.1%})')
df[['MedInc','AveRooms','bedroom_ratio','rooms_per_person','is_expensive']].head()

In [ ]:
# Chained comparisons for data validation
valid_income = df[(0.5 <= df['MedInc']) & (df['MedInc'] <= 15.0)]
valid_age    = df[(1 <= df['HouseAge']) & (df['HouseAge'] <= 52)]
print(f'Valid income rows: {len(valid_income)}/{len(df)}')
print(f'Valid age rows   : {len(valid_age)}/{len(df)}')

In [ ]:
# Bitwise operators for feature flags
import numpy as np
NUMERIC  = 0b0001
TEXT     = 0b0010
IMAGE    = 0b0100
TEMPORAL = 0b1000

for name, flags in [('model_a', NUMERIC | TEXT), ('model_b', NUMERIC | IMAGE | TEMPORAL)]:
    used = [k for k, v in dict(NUMERIC=NUMERIC,TEXT=TEXT,IMAGE=IMAGE,TEMPORAL=TEMPORAL).items() if flags & v]
    print(f'{name} ({flags:04b}): {used}')

In [ ]:
# Visualise: house value vs median income
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scatter: income vs value (operators used for colour split)
mask = df['is_expensive']  # boolean operator result used as mask
axes[0].scatter(df.loc[~mask,'MedInc'], df.loc[~mask,'MedHouseVal'], alpha=0.1, s=1, label='affordable')
axes[0].scatter(df.loc[ mask,'MedInc'], df.loc[ mask,'MedHouseVal'], alpha=0.1, s=1, color='red', label='expensive')
axes[0].set(xlabel='MedInc', ylabel='MedHouseVal', title='Income vs House Value')
axes[0].legend(markerscale=5)

# Histogram of bedroom ratio
df['bedroom_ratio'].clip(0, 1).hist(bins=40, ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set(xlabel='AveBedrms / AveRooms', ylabel='Count', title='Bedroom Ratio')
plt.tight_layout(); plt.show()